# Roots: where records go

A record type stores its records under `base / path`, joined like `pathlib`'s `/`:

- the **base root** is the innermost `with td.use_root(...)` block, else `td.set_root(...)`,
  else `$TENSORDIR_ROOT`, else the working directory;
- `path` is the `root=` argument of a call, else the class's `root` (`@td.root(path)` or the
  attribute `root = path`), else nothing. A `root=` argument replaces the class root.

A relative `path` extends the base root, an absolute one replaces it. `td.get_root(Cls)` shows
where `Cls` records go right now. This notebook checks each rule in a temporary directory.

In [ ]:
import contextvars
import os
import shutil
import tempfile
import threading
from pathlib import Path

import numpy as np

import tensordir as td

tmp = Path(tempfile.mkdtemp(prefix="tensordir-roots-")).resolve()  # resolved like the working directory
os.environ.pop("TENSORDIR_ROOT", None)
td.set_root(None)  # start from the defaults
os.chdir(tmp)  # the working directory is the last fallback


def show(path):
    """`path` relative to this notebook's temporary directory."""
    return f"<tmp>/{Path(path).relative_to(tmp)}" if Path(path) != tmp else "<tmp>"

## Three record types

`Stimulus` has no class root, `Connectome` a relative one, `Atlas` an absolute one.

In [ ]:
class Stimulus(td.Record):
    def __init__(self):
        self.frames = np.zeros((4, 8, 8))


@td.root("connectomes")  # relative: joined onto the base root
class Connectome(td.Record):
    def __init__(self):
        self.weights = np.eye(4)


class Atlas(td.Record):
    root = tmp / "shared" / "atlases"  # absolute: the base root does not matter

    def __init__(self):
        self.regions = ["V1", "LM", "AL"]


def where_now():
    for cls in (Stimulus, Connectome, Atlas):
        print(f"{cls.__name__:<11} -> {show(td.get_root(cls))}")


print("base root:", show(td.get_root()))
where_now()

## The base root, by precedence

`$TENSORDIR_ROOT` beats the working directory and is read whenever a root is resolved.

In [ ]:
os.environ["TENSORDIR_ROOT"] = str(tmp / "env")
print("base root:", show(td.get_root()))
where_now()

`td.set_root` beats `$TENSORDIR_ROOT`; a relative value is resolved against the working
directory when it is set.

In [ ]:
td.set_root("project")
print("base root:", show(td.get_root()))
where_now()

`td.use_root` beats `set_root` inside its block; blocks nest and the innermost wins. Tests use
it to redirect every record type with a relative root at once.

In [ ]:
with td.use_root(tmp / "test") as base:
    print("block:", show(base))
    where_now()
    with td.use_root(tmp / "inner"):
        print("inner block:", show(td.get_root(Connectome)))
print("after the block:", show(td.get_root()))

## The `root=` argument

`get`, `locate`, `lookup`, `query`, `info` and `td.reindex` take `root=`, which replaces the
class root and follows the same join rule.

In [ ]:
default = Connectome()
scratch = Connectome.get(root="scratch")
fast = Connectome.get(root=tmp / "fast" / "run1")
for record in (default, scratch, fast):
    print(show(record.path.parent.parent), record.key[:16])
print(len(Connectome.query()), len(Connectome.query(root="scratch")), Connectome.resolve_root("scratch") == td.get_root(Connectome, root="scratch"))

The root never enters the key: the three records share a key, and a record copied to another
root is found there. Each resolved root has its own `.tensordir/` with leases, temporary builds
and index.

In [ ]:
archive = tmp / "archive" / "connectomes" / "Connectome" / default.path.name
shutil.copytree(default.path, archive)
copied = Connectome.lookup(root=tmp / "archive" / "connectomes")
print(copied.key == default.key, show(copied.path))
print(sorted(show(p) for p in tmp.rglob(".tensordir")))

## Threads

`use_root` is context-local: it applies to the current thread or asyncio task. A thread started
inside a block resolves against `set_root`, `$TENSORDIR_ROOT` or the working directory, unless
it runs in a copy of the context.

In [ ]:
seen = {}
with td.use_root(tmp / "test"):
    plain = threading.Thread(target=lambda: seen.update(plain=td.get_root()))
    context = contextvars.copy_context()
    copied = threading.Thread(target=lambda: context.run(lambda: seen.update(copied=td.get_root())))
    plain.start(), copied.start()
    plain.join(), copied.join()
print({name: show(path) for name, path in seen.items()})

## Roots next to source files

A relative class root follows the base root. For data that belongs to a package (fixtures,
reference connectomes), write an absolute path from the module file:

```python
@td.root(Path(__file__).parent / "data")
class Fixture(td.Record): ...
```

`@td.root(None)` removes an inherited root.

In [ ]:
@td.root(None)
class LocalConnectome(Connectome):
    pass


print(show(td.get_root(Connectome)), show(td.get_root(LocalConnectome)))

In [ ]:
os.environ.pop("TENSORDIR_ROOT")  # leave the process as we found it
td.set_root(None)